# Configure

In [1]:
%load_ext autoreload
%autoreload 2

In [34]:
# Packages
import requests
import os
import re
from os.path import join
from pathlib import Path
import yaml
from yaml.loader import SafeLoader
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon
import numpy as np
import rioxarray as rio

import unsafe.download as undown
import unsafe.files as unfile
import unsafe.unzip as ununzip
import unsafe.exp as unexp
import unsafe.ddfs as unddf
import unsafe.ensemble as unens

In [3]:
# Name the fips, statefips, stateabbr, and nation that
# we are using for this analysis
# We pass these in as a list even though the framework currently
# processes a single county so that it can facilitate that
# expansion in the future
# TODO - could make sense to define these in the future
# in json or other formats instead of as input in code
fips_args = {
    'FIPS': ['42101'], 
    'STATEFIPS': ['42'],
    'STATEABBR': ['PA'],
    'NATION': ['US']
}
FIPS = fips_args['FIPS'][0]
NATION = fips_args['NATION'][0]

In [4]:
# We need to pass in a config file that sets up
# constants and the structure for downlading data
# For the directory structure of our case study, 
# we use the following 
ABS_DIR = Path().absolute().parents[0]

CONFIG_FILEP = join(ABS_DIR, 'config', 'config.yaml')
# Open the config file and load
with open(CONFIG_FILEP) as f:
    CONFIG = yaml.load(f, Loader=SafeLoader)

# Wildcards for urls
URL_WILDCARDS = CONFIG['url_wildcards']

# Get the file extensions for api endpoints
API_EXT = CONFIG['api_ext']

# Get the CRS constants
NSI_CRS = CONFIG['nsi_crs']

# Dictionary of ref_names
REF_NAMES_DICT = CONFIG['ref_names']

# Dictionary of ref_id_names
REF_ID_NAMES_DICT = CONFIG['ref_id_names']

# Coefficient of variation
# for structure values
COEF_VARIATION = CONFIG['coef_var']

# First floor elevation dictionary
FFE_DICT = CONFIG['ffe_dict']

# Number of states of the world
N_SOW = CONFIG['sows']

# Data for flood depth grids
# Get hazard model variables
HAZ_FILEN = CONFIG['haz_filename']
# Get CRS for depth grids
HAZ_CRS = CONFIG['haz_crs']
# Ensemble members
HAZ_NENS = CONFIG['haz_nens']
# Number of columns for each depth grid
HAZ_NCOLS = CONFIG['haz_ncols']
# Num rows for each depth grid
HAZ_NROWS = CONFIG['haz_nrows']
# Lower left x coordinate
HAZ_XLL = CONFIG['haz_xll']
# Lower left y coordinate
HAZ_YLL = CONFIG['haz_yll']
# Cell resolution
HAZ_RES = CONFIG['haz_res']
# NODATA values
HAZ_NODATA = CONFIG['haz_nodata']

# Get the files we need downloaded
DOWNLOAD = pd.json_normalize(CONFIG['download'], sep='_').T

# We can also specify the filepath to the
# raw data directory
FR = join(ABS_DIR, "data", "raw")

# And external - where our hazard data should be
FE = join(FR, "external")

# Set up interim and results directories as well
# We already use "FR" for raw, we use "FO" 
# because you can also think of results
# as output
FI = join(ABS_DIR, "data", "interim")
FO = join(ABS_DIR, "data", "results")

# "Raw" data directories for exposure, vulnerability (vuln) and
# administrative reference files
EXP_DIR_R = join(FR, "exp")
VULN_DIR_R = join(FR, "vuln")
REF_DIR_R = join(FR, "ref")
# Haz is for depth grids
HAZ_DIR_R = join(FE, "haz")
# Pol is for NFHL
POL_DIR_R = join(FR, "pol")

# Unzip directory 
UNZIP_DIR = join(FR, "unzipped")

# We want to process unzipped data and move it
# to the interim directory where we keep
# processed data
# Get the filepaths for unzipped data
# We unzipped the depth grids (haz) and 
# ddfs (vuln) into the "external"/ subdirectory
HAZ_DIR_UZ = join(UNZIP_DIR, "external", "haz")
POL_DIR_UZ = join(UNZIP_DIR, "pol")
REF_DIR_UZ = join(UNZIP_DIR, "ref")
VULN_DIR_UZ = join(UNZIP_DIR, "external", "vuln")

# "Interim" data directories
EXP_DIR_I = join(FI, "exp")
VULN_DIR_I = join(FI, "vuln")
REF_DIR_I = join(FI, "ref")
# Haz is for depth grids
HAZ_DIR_I = join(FI, "haz")
# Pol is for NFHL
POL_DIR_I = join(FI, "pol")

# Download and unzip data

In [5]:
wcard_dict = {x: fips_args[x[1:-1]][0] for x in URL_WILDCARDS}
undown.download_raw(DOWNLOAD, wcard_dict,
                    FR, API_EXT)

Downloaded from: https://nsi.sec.usace.army.mil/nsiapi/structures?fips=42101
Downloaded from: https://phl.carto.com/api/v2/sql?filename=opa_properties_public&format=geojson&skipfields=cartodb_id&q=SELECT+*+FROM+opa_properties_public
Downloaded from: https://opendata.arcgis.com/api/v3/datasets/ab9e89e1273f445bb265846c90b38a96_0/downloads/data?format=geojson&spatialRefId=4326&where=1%3D1
Downloaded from: https://opendata.arcgis.com/api/v3/datasets/84baed491de44f539889f2af178ad85c_0/downloads/data?format=geojson&spatialRefId=4326&where=1%3D1
Downloaded from: https://hazards.fema.gov/nfhlv2/output/County/420757_20230701.zip
Downloaded from: https://www2.census.gov/geo/tiger/TIGER2022/TRACT/tl_2022_42_tract.zip
Downloaded from: https://www2.census.gov/geo/tiger/TIGER2022/BG/tl_2022_42_bg.zip
Downloaded from: https://www2.census.gov/geo/tiger/TIGER2022/TABBLOCK20/tl_2022_42_tabblock20.zip
Downloaded from: https://static-data-screeningtool.geoplatform.gov/data-versions/1.0/data/score/download

In [6]:
ununzip.unzip_raw(FR, UNZIP_DIR)

Unzipped: nfhl
Unzipped: zcta
Unzipped: county
Unzipped: bg
Unzipped: tract
Unzipped: block
Unzipped: ddfs
Unzipped: RIFT_domain
Unzipped: Irene


# Prepare data for ensemble

The study domain corresponds to 12 digit USGS hydrological unit code (HUC) watershed 020402031008. We will spatially merge the NSI structures and Philadelphia data to this extent. We will restrict the other downloaded geospatial data to objects that intersect with this (e.g., Census Tracts that overlap). We may clip these for plotting purposes later.

## Study area boundary

In [5]:
CLIP_SHP_FILEP = join(HAZ_DIR_UZ, 'RIFT_domain', 'domain_1.shp')
clip_geo = gpd.read_file(CLIP_SHP_FILEP)

/Users/f006dwr/miniforge3/envs/nsi_fit/lib/python3.12/site-packages/pyogrio/geopandas.py:49: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  res = pd.to_datetime(ser, **datetime_kwargs)


## Process exposure

We will start by merging several datasets to the envelope of our clip polygon and then do the processing on those subsets.

### Get subsets of NSI and Philly data

In [122]:
# Load in the NSI and Philly assessor, parcel, and footprint data
nsi_gdf = unexp.get_nsi_geo(FIPS, NSI_CRS, EXP_DIR_R)

assess_cols = ['assessment_date', 'basements', 'building_code',
               'building_code_description', 'building_code_description_new',
               'category_code', 'category_code_description', 'census tract',
               'exterior_condition', 'garage_type', 'general_construction',
               'interior_condition','location', 'market_value',
               'market_value_date', 'number_stories', 'owner_1',
               'parcel_number', 'sale_date', 'sale_price',
               'quality_grade', 'taxable_building', 'taxable_land',
               'topography', 'unit', 'year_built',
               'year_built_estimate', 'zoning']
assess = gpd.read_file(join(EXP_DIR_R, FIPS, 'assess.geojson'),
                       mask=clip_geo, columns=assess_cols)

parcel = gpd.read_file(join(EXP_DIR_R, FIPS, 'parcel.geojson'),
                       mask=clip_geo)
bld_fp = gpd.read_file(join(EXP_DIR_R, FIPS, 'bldfp.geojson'),
                       mask=clip_geo)

Prepared geodataframe


/Users/f006dwr/miniforge3/envs/nsi_fit/lib/python3.12/site-packages/pyogrio/geopandas.py:49: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  res = pd.to_datetime(ser, **datetime_kwargs)
/Users/f006dwr/miniforge3/envs/nsi_fit/lib/python3.12/site-packages/pyogrio/geopandas.py:49: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  res = pd.to_datetime(ser, **datetime_kwargs)


#### NSI subset

We'll follow existing UNSAFE functions to get our NSI dataset. We're going to include any residential structure up to 2 stories whereas we previously only looked at single family households. 

In [230]:
# Set the values that we pass into the get_struct_subset function
occtype_list=['RES1-1SNB', 'RES1-2SNB', 'RES1-1SWB', 'RES1-2SWB',
              'RES1-SLNB', 'RES1-SLWB', 'RES1-3SNB', 'RES1-3SWB',
              'RES3A', 'RES3B', 'RES3C', 'RES3D', 'RES3E', 'RES3F']
sub_string = 'occtype.isin(@occtype_list)'
nsi_filt = unexp.get_struct_subset(nsi_gdf,
                                   filter=sub_string,
                                   occtype_list=occtype_list)

EXP_OUT_FILEP = join(EXP_DIR_I, FIPS, 'nsi_res.pqt')
unfile.prepare_saving(EXP_OUT_FILEP)

# Clip to our boundary to reduce file size
nsi_clip_out = gpd.clip(nsi_filt, clip_geo.to_crs(nsi_filt.crs))

# Write file
nsi_clip_out.to_parquet(EXP_OUT_FILEP, index=False)


In [231]:
len(nsi_clip_out)

94824

#### Philly subsets

We want to use the assessment data to identify residential structures up to two stories. Then we will subset the building footprints and parcels correspondingly. To match up records, we will link `assess['parcel_number']` to `parcel['BRT_ID']` to `bld_fp['PARCEL_ID_NUM']`.

From the Maps@Phila.gov email: “The buildings are matched via their centroid to the PWD Parcels for their parcelid, they could use the parcelid to connect to PWD Parcels, then use the BRT_ID field in the PWD Parcels to get to the OPA Tax Accounts.  This won’t be the cleanest solution for condos, but there’s no real system for handling those anywhere.  You can tell [redacted] she’s welcome to point out any mismatches she finds directly to me, I’ve worked with her before on other projects.”

In [123]:
def split_bld_code(bld_desc):

    """
    Split a building code description into tokens based on the first numeric value.

    This function takes a building code description and splits it into tokens where
    all text before the first numeric value becomes one token, and all subsequent
    words (including numeric values) become individual tokens.

    Parameters
    ----------
    bld_desc : str
        A string containing the building code description.
        Example: 'APT 2-4 UNITS 3.5 STY MAS'

    Returns
    -------
    list
        A list where the first element is all text before the first number (as one string),
        followed by all remaining words as individual elements.
        Example: ['APT', '2-4', 'UNITS', '3.5', 'STY', 'MAS']
        If no numeric values are found, returns the entire description as a single element list.

    Examples
    --------
    >>> split_bld_code('APT 2-4 UNITS 3.5 STY MAS')
    ['APT', '2-4', 'UNITS', '3.5', 'STY', 'MAS']
    
    >>> split_bld_code('DET W/GAR 2 STY MASONRY')
    ['DET W/GAR', '2', 'STY', 'MASONRY']
    """

    if bld_desc is None:
        return bld_desc

    # Split the building code description into words
    full_code = bld_desc.split()

    # Find the index of the first string with a number as first character
    first_num_idx = next((i for i, word in enumerate(full_code) if word[0].isdigit()), None)
    
    if first_num_idx is not None:
        # Join everything before the first number as one token
        prefix = ' '.join(full_code[:first_num_idx])
        # Keep remaining words as separate tokens
        remaining = full_code[first_num_idx:]
        return [prefix] + remaining
    else:
        return [' '.join(full_code)]

In [124]:
# We want to retain structures with a building code description
assess_sub = assess[assess['building_code_description'].notnull()].copy()
# split up the building code description field
assess_sub.loc[:, 'bld_code_split'] = assess_sub['building_code_description'].apply(split_bld_code)

# get the occupancy type code and the remaining token 
# into separate columns
assess_sub.loc[:, 'bld_type'] = assess_sub['bld_code_split'].apply(lambda x: x[0])
assess_sub.loc[:, 'bld_code_rest'] = assess_sub['bld_code_split'].apply(lambda x: x[1:])
# helpful to have the rest as a single string for some inspections
# can drop the last token though (usually foundation type)
assess_sub.loc[:, 'bld_code_rest_str'] = assess_sub['bld_code_rest'].apply(lambda x: ' '.join(x[:-1]))

# We want to subset to the category codes that may have res buildings
cat_codes = ['1', '2', '3', '14']
assess_sub = assess_sub[assess_sub['category_code'].str.strip().isin(cat_codes)]

# We do not want "VACANT" 
assess_sub = assess_sub[~assess_sub['bld_type'].str.contains('VACANT')]

# We can also drop anything with empty bld_code_rest_str
assess_non_res = assess_sub[assess_sub['bld_code_rest_str'] == '']
assess_sub = assess_sub[assess_sub['bld_code_rest_str'] != '']

In [125]:
# sample a few records from each bld_type group
samples = assess_sub.groupby('bld_type').apply(lambda x: x.sample(n=min(10, len(x)))).reset_index(drop=True)
# write out the parcel numbers and a few other columns and start 
# checking the ddf pairing
check_cols = ['parcel_number', 'bld_type', 'bld_code_rest_str',
              'building_code', 'category_code_description', 'zoning']
check_filep = join(EXP_DIR_I, 'check_codes_020425.csv')
samples[check_cols].to_csv(check_filep, index=False)

/var/folders/d2/g0h08s551zb2hz_ws2g4ggh400hbd0/T/ipykernel_2353/3102070186.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  samples = assess_sub.groupby('bld_type').apply(lambda x: x.sample(n=min(10, len(x)))).reset_index(drop=True)


In [140]:
assess_sub[assess_sub['parcel_number'] == '342136901'][check_cols + ['building_code_description_new', 'market_value_date', 'assessment_date']]

,parcel_number,bld_type,bld_code_rest_str,building_code,category_code_description,zoning,building_code_description_new,market_value_date,assessment_date
28311,342136901,DET,2.5 STY,A41,SINGLE FAMILY,RSD3,COLONIAL,None,2024-06-06 16:08:48+00:00


In [141]:
assess_sub[assess_sub['parcel_number'] == '405756151'][check_cols + ['building_code_description_new', 'market_value_date', 'assessment_date']]

,parcel_number,bld_type,bld_code_rest_str,building_code,category_code_description,zoning,building_code_description_new,market_value_date,assessment_date
64903,405756151,DET,1 STY,A16,SINGLE FAMILY,RSA3,None,None,2024-06-06 16:03:03+00:00


In [242]:
# First, subset assessments to residential (this is based on 
# building code mappings we identified externally)
# We will separate out condos, apartments, single/multifamily res
res_bld_codes = []
assess_res = assess[assess['building_code'].isin(res_bld_codes)].drop(columns='geometry')

# We want to take out the condos from this dataframe
# Condos need to be aggregated by address and then
# linked by address to parcels, whereas other residential
# structures can be linked by parcel_number/BRT_ID
# We also want to aggregate the structure value
# of the condo units before merging

# We also want to take out apartments from here because we will 
# divide their structure values by the number of building footprints
# they get linked to

# Second merge assess & pwd parcels
assess_par = assess_res.merge(parcel.drop(columns='geometry'),
                              left_on=['parcel_number'],
                              right_on=['BRT_ID'],
                              how='inner')

# There will be some assessment records that don't merge because 
# they are condos. We take condos that don't merge in
# and do a link on address to get them into the assess/parcel
# dataframe. 

# Next, merge assess_parcel w/ bld_fp
assess_bld = assess_par.merge(bld_fp,
                              left_on='PARCEL_ID',
                              right_on='PARCEL_ID_NUM',
                              how='inner')

# There are some entries that do not successfully merge
# so we need to figure out how to deal with these

# We also need to review the resulting assessment/bld_fp
# matches to build confidence they are accurate. In particular,
# we want to make transparent and quality-assured choices
# about restricting the structure sample to 
# the kinds of residential structures for which
# we can estimate damages

In [247]:
assess_bld[assess_bld['location'] == '3400 W SCHOOL HOUSE LN'].iloc[:,30:45]

,BRT_ID,NUM_BRT,NUM_ACCOUNTS,GROSS_AREA,PIN,PARCEL_ID,Shape__Area,Shape__Length,BIN,PARCEL_ID_NUM,geometry
1,383108105,1.0,2,176786,1.001475e+09,535489,28036.398438,759.250396,1502999,535489,"POLYGON ((-75.19318 40.01826, -75.19311 40.018..."
2,383108105,1.0,2,176786,1.001475e+09,535489,28036.398438,759.250396,1525389,535489,"POLYGON ((-75.19399 40.01821, -75.19396 40.018..."
3,383108105,1.0,2,176786,1.001475e+09,535489,28036.398438,759.250396,1536764,535489,"POLYGON ((-75.19394 40.01845, -75.19399 40.018..."


## Process vulnerability

## Process reference data

## Hazard

In [ ]:
# HAZ_DIR_UZ
# HAZ_FILEN (need to modify with a wildcard for ensemble numbers from 01 to 50 - haz_nens)
# We need a function that turns any of the files into a depth grid
# We don't necessarily have to save these - might not want all that data
# I do want to do this for the "best estimate" though
# I only want the array part of the others
HAZ_CRS

In [ ]:
HAZ_DIR_UZ